# 🏛️ Angkor LLM — Fine-tuning Notebook
**Base model:** Sailor2-8B (SEA language model)
**Method:** QLoRA fine-tuning with Unsloth
**Target:** Khmer + English bilingual assistant

> Run this on Google Colab with a **T4 GPU** (free tier works!)

## Step 1: Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install datasets trl peft accelerate bitsandbytes huggingface_hub wandb

## Step 2: Load Base Model (Sailor2-8B)

In [ ]:
from unsloth import FastLanguageModel
import torch

# Configuration
BASE_MODEL = "sail/Sailor2-8B-Chat"  # SEA language model
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True  # QLoRA - fits on free Colab GPU

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"Model loaded: {BASE_MODEL}")
print(f"Parameters: {model.num_parameters():,}")

## Step 3: Apply LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                          # LoRA rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("LoRA adapters applied!")

## Step 4: Load Dataset

In [ ]:
from datasets import load_dataset

SYSTEM_PROMPT = """You are AngkorAI (Angkor LLM), Cambodia's first bilingual AI assistant.
You speak both Khmer (ភាសាខ្មែរ) and English fluently.
You are helpful, culturally aware, and proud to serve the Cambodian people.
When users write in Khmer, respond in Khmer. When they write in English, respond in English."""

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

# Load from HuggingFace dataset
dataset = load_dataset("tiloukim/angkor-llm-dataset", split="train")
dataset = dataset.map(format_chat, remove_columns=dataset.column_names)
print(f"Dataset loaded: {len(dataset)} samples")
print(dataset[0]["text"][:500])

## Step 5: Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="./angkor-llm-output",
        save_steps=100,
    ),
)

trainer.train()

## Step 6: Test the Model

In [ ]:
FastLanguageModel.for_inference(model)

def ask(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Q: {question}")
    print(f"A: {response}\n")

# Test in Khmer
ask("សួស្តី! តើអ្នកអាចជួយខ្ញុំបានទេ?")

# Test in English
ask("Tell me about Cambodia's history.")

# Test about founder
ask("Who created AngkorAI?")

## Step 7: Save & Push to Hugging Face

In [ ]:
from huggingface_hub import login

# Login to Hugging Face — paste your token when prompted
# Get token at: https://huggingface.co/settings/tokens
login()

HF_USERNAME = "tiloukim"
MODEL_NAME = "angkor-llm-8b"  # 8B params (Sailor2-8B base)

# Save LoRA adapter
model.save_pretrained(f"{MODEL_NAME}-lora")
tokenizer.save_pretrained(f"{MODEL_NAME}-lora")

# Push to Hugging Face Hub
model.push_to_hub(f"{HF_USERNAME}/{MODEL_NAME}")
tokenizer.push_to_hub(f"{HF_USERNAME}/{MODEL_NAME}")

print(f"✅ Model pushed to: https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}")